In [1]:
import sys
import os

# Get the absolute path to the project directory
project_dir = os.path.abspath("..")

# Append the project directory to sys.path
if project_dir not in sys.path:
    sys.path.append(project_dir)
    
from src.predictionModule.LoadupSamples import LoadupSamples
from src.predictionModule.FilterSamples import FilterSamples
from src.predictionModule.MachineModels import MachineModels

import numpy as np
import pandas as pd
import polars as pl
import datetime
import scipy
import matplotlib.pyplot as plt
import optuna
import random
import torch

from itertools import product
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import logging
formatted_date = datetime.datetime.now().strftime("%d%b%y_%H%M").lower()

logger = logging.getLogger()
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler(sys.stdout)
formatter = logging.Formatter(fmt="%(asctime)s - %(message)s")
handler.setFormatter(formatter)
if not logger.hasHandlers():
    logger.addHandler(handler)
else:
    logger.handlers[:] = [handler]

#Output File handler
formatted_str = f"notebook-stomp-{formatted_date}"
file_handler = logging.FileHandler(f"{formatted_str}.log", mode="w")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Usage
logger.setLevel(logging.INFO)
logger.info("This will print to the notebook's output cell")

c:\Users\KILightTouch\Desktop\RandomOdyssey\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-09-09 17:28:48,854 - This will print to the notebook's output cell


In [2]:
params = {
    "idxAfterPrediction": 5,
    'timesteps': 90,
    'target_option': 'last',
    "LoadupSamples_time_scaling_stretch": True,
    "LoadupSamples_time_inc_factor": 61,

    "FilterSamples_q_up": 0.985,
    
    "FilterSamples_cat_over20": True,
    "FilterSamples_cat_posOneYearReturn": False,
    "FilterSamples_cat_posFiveYearReturn": False,

    "LSTM_val_split": 0.1,
}

In [4]:
timegroup = "group_regOHLCV_over5years"
treegroup = "group_debug"

eval_date = datetime.date(year=2025, month=7, day=13)
evaldates = [eval_date - datetime.timedelta(days=i) for i in range(1, 6)]
start_train_date = datetime.date(year=2019, month=1, day=1)
split_Date = datetime.date(year=2025, month=1, day=1)
ls = LoadupSamples(
    train_start_date=start_train_date,
    test_dates=evaldates,
    treegroup=treegroup,
    timegroup=timegroup,
    params=params,
)
ls.load_samples(main_path = "../src/featureAlchemy/bin/")
ls.split_dataset(
    start_date=start_train_date,
    last_train_date=split_Date,
    last_test_date=eval_date
)
fs_pre = FilterSamples(
    Xtree_train = ls.train_Xtree, 
    ytree_train = ls.train_ytree, 
    treenames   = ls.featureTreeNames,
    Xtree_test  = ls.test_Xtree,  
    ytree_test  = ls.test_ytree,
    meta_train  = ls.meta_pl_train, 
    meta_test   = ls.meta_pl_test, 
    params      = params
)
mask_train_pre, mask_test_pre = fs_pre.categorical_masks()
ls.apply_masks(mask_train_pre, mask_test_pre)

2025-09-09 17:29:40,816 - Test date 2025-07-12 not found in the database. Omitting.


In [5]:
Xtree_train = ls.train_Xtree
ytree_train = ls.train_ytree
Xtree_test  = ls.test_Xtree
ytree_test  = ls.test_ytree

Xtime_train = ls.train_Xtime
ytime_train = ls.train_ytime
Xtime_test  = ls.test_Xtime
ytime_test  = ls.test_ytime

treenames   = ls.featureTreeNames
timenames   = ls.featureTimeNames
meta_train  = ls.meta_pl_train
meta_test   = ls.meta_pl_test

dates_tr = meta_train['date'].unique().sort()
dates_te = meta_test['date'].unique().sort()

from src.common.DataFrameTimeOperations import DataFrameTimeOperations as dfta
dates_tr_idx = dfta(meta_train, 'date').getNextLowerOrEqualIndices(dates_tr)
dates_te_idx = dfta(meta_test, 'date').getNextLowerOrEqualIndices(dates_te)

assert not any([i == -1 for i in dates_tr_idx])
assert not any([i == -1 for i in dates_te_idx])

In [6]:
# ---- knobs (use existing globals if present) ----
BASE_RS = 42
KM_INIT = "k-means++"
device = "cuda" if torch.cuda.is_available() else "cpu"
n_splits = 5
n_test_days = 60

assert "Xtime_train" in globals() and "ytree_train" in globals(), "Need Xtime_train/ytree_train"
nS, nT, nF = Xtime_train.shape
ytr_arr = np.asarray(ytree_train)
ytr_vec = ytr_arr[:, -1] if ytr_arr.ndim == 2 else ytr_arr

yte_arr = np.asarray(ytree_test)
yte_vec = yte_arr[:, -1] if yte_arr.ndim == 2 else yte_arr

In [12]:
def geometric_mean_safe(arr):
    arr = np.asarray(arr, dtype=float)
    minv = np.min(arr) if arr.size else 0.0
    shift = -minv + 1e-9 if minv <= 0 else 0.0
    return float(np.exp(np.mean(np.log(arr + shift)))) if arr.size else np.nan

def metric(arr):
    """Custom cluster score function."""
    gm = geometric_mean_safe(arr)
    return gm - 1

def make_design(X, t_win, feat_idx):
    feat_idx = np.atleast_1d(feat_idx)
    Xw = X[:, -t_win:, feat_idx]
    return Xw.reshape(Xw.shape[0], -1)

def _score_once_lstm(
    t_win: int,
    k: int,
    f_idcs: list[int],
    s_tr_l: int,
    s_tr_u: int,
    s_te_l: int,
    s_te_u: int,
    mm: MachineModels,
    quantile_val: float = 0.9,
    do_transform: bool = False,
    device: str = "cpu",
    min_cluster_train: int = 50,
) -> float:
    """
    Cluster train window, train one LSTM per cluster, pick cluster with lowest RMSE,
    predict on the paired test samples in that cluster, and select the highest
    predicted entries via a quantile threshold.
    """

    Xtr, ytr = Xtime_train[s_tr_l:s_tr_u], ytr_vec[s_tr_l:s_tr_u]
    Xte, yte = Xtime_train[s_te_l:s_te_u], ytr_vec[s_te_l:s_te_u]

    if k >= Xtr.shape[0]:
        return -np.inf

    # design matrices
    Xd_tr = make_design(Xtr, t_win, f_idcs)
    Xd_te = make_design(Xte, t_win, f_idcs)

    if do_transform:
        scaler = StandardScaler().fit(Xd_tr)
        Xd_tr = scaler.transform(Xd_tr)
        Xd_te = scaler.transform(Xd_te)

    n_feat = len(np.atleast_1d(f_idcs))
    Xseq_tr = Xd_tr.reshape(-1, t_win, n_feat)
    Xseq_te = Xd_te.reshape(-1, t_win, n_feat)

    km = KMeans(n_clusters=k, n_init=k, init=KM_INIT).fit(Xd_tr)
    lab_tr = km.labels_
    lab_te = km.predict(Xd_te)

    best_c, best_model, best_rmse = None, None, float("inf")
    for c in range(k):
        idx_tr = np.where(lab_tr == c)[0]
        if idx_tr.size < min_cluster_train:
            logger.info(f"[LSTM] cluster {c}: skipped (train size {idx_tr.size} < {min_cluster_train})")
            continue

        Xc_tr = Xseq_tr[idx_tr]
        yc_tr = ytr[idx_tr]

        try:
            model_c, info = mm.run_LSTM_torch(
                X_train=Xc_tr,
                y_train=yc_tr,
                X_test=None,       # rely on mm.params['LSTM_val_split'] for internal validation
                y_test=None,
                device=device
            )
            val_rmse = float(info.get("val_rmse", float("inf")))
            logger.info(f"[LSTM] cluster {c}: train={idx_tr.size}, val_rmse={val_rmse:.6f}")

            if np.isfinite(val_rmse) and val_rmse < best_rmse:
                best_c, best_model, best_rmse = c, model_c, val_rmse
        except Exception as e:
            logger.exception(f"[LSTM] cluster {c}: training failed — {e}")
            continue

    if best_c is None or best_model is None:
        logger.warning("[LSTM] no trainable cluster found.")
        return -np.inf

    # predict on test members of best cluster
    idx_te_local = np.where(lab_te == best_c)[0]
    if idx_te_local.size == 0:
        logger.info(f"[LSTM] best cluster {best_c} has no test members.")
        return -np.inf

    preds = mm.predict_LSTM_torch(best_model, Xseq_te[idx_te_local], device=device)
    if preds.size == 0 or not np.all(np.isfinite(preds)):
        return -np.inf

    thr = float(np.quantile(preds, quantile_val))
    mask = preds >= thr
    y_selected = yte[idx_te_local][mask]
    y_selected = y_selected[np.isfinite(y_selected)] 

    if y_selected.size == 0:
        return -np.inf

    score = metric(y_selected)

    logger.info(
        f"[LSTM] t_win={t_win}, k={k}, tr_size={Xd_tr.shape[0]}, te_size={Xd_te.shape[0]} "
        f"-> best_c={best_c}, best_val_rmse={best_rmse:.6f}, "
        f"cluster_te={idx_te_local.size}, thr@q={quantile_val}={thr}, score={score}"
    )

    return float(score) if np.isfinite(score) else -np.inf

In [13]:
max_training_days = 1200
N = len(dates_tr_idx)
lo = max_training_days - 1                      # min pivot (last train index)
hi = N - n_test_days - 1                      # max pivot
eligible = list(range(lo, hi + 1))
assert len(eligible) >= n_splits, f"Too few eligible pivots ({len(eligible)}) for n_splits={n_splits}"
pivots = sorted(random.sample(eligible, n_splits))

tr_slice_list = [slice(p - max_training_days + 1, p + 1) for p in pivots]      # [..)
te_slice_list = [slice(p + 1, p + 1 + n_test_days) for p in pivots]          # [..)

for p in pivots:
    logger.info(f"  Pivot {p}: Date {dates_tr[p]}")

2025-09-09 17:31:52,263 -   Pivot 1263: Date 2024-01-09
2025-09-09 17:31:52,264 -   Pivot 1280: Date 2024-02-02
2025-09-09 17:31:52,265 -   Pivot 1288: Date 2024-02-14
2025-09-09 17:31:52,266 -   Pivot 1311: Date 2024-03-19
2025-09-09 17:31:52,266 -   Pivot 1441: Date 2024-09-24


In [14]:
def make_objective():
    def objective(trial: optuna.Trial) -> float:
        # search space
        t_win = trial.suggest_int("TIME_WINDOW", 5, 25, step = 5)
        k = trial.suggest_int("N_CLUSTERS", 3, 18, step = 5)
        n_training_days = trial.suggest_int("N_TRAIN_DAYS", 800, 900, step = 50)
        do_transform = False
        f_idcs_cat = 2
        quantile_val = 0.9

        opt_params = params.copy()

        if f_idcs_cat == 0:       f_idcs = [0]
        elif f_idcs_cat == 1:     f_idcs = [1]
        elif f_idcs_cat == 2:     f_idcs = [0, 1]

        scores = []

        mm = MachineModels(params=opt_params)
        for i in range(n_splits):
            tr_idx = tr_slice_list[i]
            te_idx = te_slice_list[i]

            # example: get date bounds if needed
            s_tr_l, s_tr_u = dates_tr_idx[tr_idx.stop - n_training_days + 1], dates_tr_idx[tr_idx.stop] - 1
            s_te_l, s_te_u = dates_tr_idx[te_idx.start], dates_tr_idx[te_idx.stop] - 1

            try:
                sc = _score_once_lstm(t_win, k, f_idcs, s_tr_l, s_tr_u, s_te_l, s_te_u, mm, 
                    device=device, do_transform=do_transform, quantile_val=quantile_val)
            except Exception as e:
                logger.info(f"Exception during scoring: {e}")
                sc = -np.inf
            scores.append(sc)

        vals = [v for v in scores if np.isfinite(v)]
        vals = np.array(vals)
        vals_log = np.log(1.0 + vals)
        if len(vals) < (len(scores)//2):
            return 0.0
        return float(np.mean(vals_log)) if len(vals_log) else -np.inf
    return objective

In [15]:
studytime = 60*60*1
n_startup_trials = 5
studyname = f"optuna_clustering_idea_{formatted_str}"

In [16]:
# === Optuna driver ===
optuna.logging.enable_propagation()
sampler = optuna.samplers.TPESampler(n_startup_trials=n_startup_trials)
study = optuna.create_study(
    study_name=studyname,
    storage="sqlite:///sandbox_optuna.db",
    direction="maximize",
    load_if_exists=True,
    sampler=sampler,
)
study.optimize(make_objective(), timeout=studytime)

logger.info(f"Best parameters: {study.best_params}")
logger.info(f"Best score: {study.best_value}")

df: pd.DataFrame = study.trials_dataframe()
logger.info("\nTrials DataFrame:")
logger.info(df.sort_values("value").to_string())

param_importances = optuna.importance.get_param_importances(study)
logger.info("Parameter Importances:")
for key, value in param_importances.items():
    logger.info(f"{key}: {value}")

[I 2025-09-09 17:31:52,382] Using an existing study with name 'optuna_clustering_idea_notebook-stomp-09sep25_1728' instead of creating a new one.


2025-09-09 17:31:52,382 - Using an existing study with name 'optuna_clustering_idea_notebook-stomp-09sep25_1728' instead of creating a new one.


Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

2025-09-09 17:31:53,419 - Epoch 1/10 — Train RMSE: 0.9211 — Validation RMSE: 0.9062


Epochs:  10%|█         | 1/10 [00:00<00:01,  4.72it/s]

...

Epochs:  60%|██████    | 6/10 [00:04<00:03,  1.28it/s]
[W 2025-09-09 17:35:16,153] Trial 9 failed with parameters: {'TIME_WINDOW': 15, 'N_CLUSTERS': 3, 'N_TRAIN_DAYS': 900} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\KILightTouch\Desktop\RandomOdyssey\.venv\Lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\KILightTouch\AppData\Local\Temp\ipykernel_9684\1379414267.py", line 29, in objective
    sc = _score_once_lstm(t_win, k, f_idcs, s_tr_l, s_tr_u, s_te_l, s_te_u, mm,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\KILightTouch\AppData\Local\Temp\ipykernel_9684\1537524234.py", line 71, in _score_once_lstm
    model_c, info = mm.run_LSTM_torch(
                    ^^^^^^^^^^^^^^^^^^
  File "c:\Users\KILightTouch\Desktop\RandomOdyssey\src\predictionModule\MachineModels.py", li

2025-09-09 17:35:16,153 - Trial 9 failed with parameters: {'TIME_WINDOW': 15, 'N_CLUSTERS': 3, 'N_TRAIN_DAYS': 900} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\KILightTouch\Desktop\RandomOdyssey\.venv\Lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\KILightTouch\AppData\Local\Temp\ipykernel_9684\1379414267.py", line 29, in objective
    sc = _score_once_lstm(t_win, k, f_idcs, s_tr_l, s_tr_u, s_te_l, s_te_u, mm,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\KILightTouch\AppData\Local\Temp\ipykernel_9684\1537524234.py", line 71, in _score_once_lstm
    model_c, info = mm.run_LSTM_torch(
                    ^^^^^^^^^^^^^^^^^^
  File "c:\Users\KILightTouch\Desktop\RandomOdyssey\src\predictionModule\MachineModels.py", line 575, in run_LSTM_torch
    loss.backward()
  File "c:\

[W 2025-09-09 17:35:16,157] Trial 9 failed with value None.


2025-09-09 17:35:16,157 - Trial 9 failed with value None.


KeyboardInterrupt: 